In [0]:
-- county sales performance
CREATE OR REPLACE MATERIALIZED VIEW gold_county_performance
AS
SELECT
    s.county,
    d.month_num,
    d.quarter,
    COUNT(DISTINCT f.store_id)  AS store_count,
    COUNT(*)                    AS transactions,
    SUM(f.sale_dollars)         AS total_revenue,
    SUM(f.bottles_sold)         AS total_bottles,
    ROUND(SUM(f.sale_dollars) / NULLIF(COUNT(DISTINCT f.store_id), 0), 2)
                                AS revenue_per_store
FROM dev.silver.fct_sales_deduped f
JOIN dev.silver.dim_stores s
    ON  f.store_id = s.store_id
    AND f.sale_date >= s.__START_AT
    AND (f.sale_date < s.__END_AT OR s.__END_AT IS NULL)
JOIN dev.silver.dim_dates d
    ON f.sale_date = d.date_day
GROUP BY s.county, d.month_num, d.quarter

In [0]:
-- city sales rankings

CREATE OR REPLACE MATERIALIZED VIEW gold_city_rankings
AS
SELECT
    s.city,
    s.county,
    COUNT(DISTINCT f.store_id)  AS store_count,
    COUNT(*)                    AS transactions,
    SUM(f.sale_dollars)         AS total_revenue,
    SUM(f.bottles_sold)         AS total_bottles,
    RANK() OVER (ORDER BY SUM(f.sale_dollars) DESC) AS city_rank
FROM dev.silver.fct_sales_deduped f
JOIN dev.silver.dim_stores s
    ON  f.store_id = s.store_id
    AND f.sale_date >= s.__START_AT
    AND (f.sale_date < s.__END_AT OR s.__END_AT IS NULL)
GROUP BY s.city, s.county